# Analysis based on DL-Hard Dataset

This part is based on the dataset called DL-Hard, which is an annotated dataset buiilt upon TREC Deep Learning questions. The questions are annotated with query intent categories based on the paper An Intent Taxonomy for Questions Asked in Web Search (https://marksanderson.org/files/papers/CHIIR21b.pdf) and answer type which is a manual annotation of target answer type for we questions (short answer, factoid etc). They also identified 50 hard questions from the ~100 TREC DL 2019/2020 dataset, which we use to compare QE technique performance on hard and non-hard queries. 

## Grouping queries based on query intent and answer type

In [1]:
import pyterrier as pt
import pandas as pd
dataset = pt.get_dataset("msmarco_passage")
index = dataset.get_index(variant="terrier_stemmed")

# Load DL-hard annotations
annotations_url = "https://raw.githubusercontent.com/grill-lab/DL-Hard/main/annotations/query/annotations.tsv"
annotations = pd.read_csv(
    annotations_url,
    sep="\t",
    header=None,
    names=["qid", "query", "intent", "answer_type", "topic_domain", "serp_type"],
)
annotations["qid"] = annotations["qid"].astype(str)

# Load TREC DL 2019/2020 qrels. This dataset contains relevance judgements from scale 0 to 3.
dl19 = pt.get_dataset("irds:msmarco-passage/trec-dl-2019")
dl20 = pt.get_dataset("irds:msmarco-passage/trec-dl-2020")
all_qrels = pd.concat([dl19.get_qrels(), dl20.get_qrels()])
all_qrels["qid"] = all_qrels["qid"].astype(str)

# Keep only queries that have both annotations and judgements.
qrels_qids = set(all_qrels["qid"].unique())
topics = annotations[annotations["qid"].isin(qrels_qids)].reset_index(drop=True)
print(f"Queries with both annotations and relevance values: {len(topics)}")
print(topics["intent"].value_counts())
print("\n")
print(topics["answer_type"].value_counts())

Queries with both annotations and relevance values: 97
intent
description     48
list            10
quantity         9
reason           8
attribute        7
entity           5
verification     3
process          3
language         3
weather          1
Name: count, dtype: int64


answer_type
factoid              28
definition           21
long answer          16
list                 10
short description     7
comparison            5
short answer          5
guide                 2
weather               1
Long answer           1
multi-answer          1
Name: count, dtype: int64


In [2]:
# Answer type can be further divided instead of keeping 12 categories
intent_map = {
    "description": "Informational",
    "reason": "Informational",
    "process": "Informational",
    "entity": "Fact-Seeking",
    "attribute": "Fact-Seeking",
    "verification": "Fact-Seeking",
    "language": "Fact-Seeking",
    "list": "Data/List",
    "quantity": "Data/List",
    "weather": "Data/List",
}

answer_map = {
    "definition": "Narrative",
    "long answer": "Narrative",
    "Long answer": "Narrative",
    "short description": "Narrative",
    "guide": "Narrative",
    "factoid": "Atomic",
    "short answer": "Atomic",
    "weather": "Atomic",
    "list": "Complex",
    "comparison": "Complex",
    "multi-answer": "Complex",
}
topics["meta_intent"] = topics["intent"].map(intent_map)
print(topics["meta_intent"].value_counts(), "\n")
topics["meta_answer_type"] = topics["answer_type"].map(answer_map)
print(topics["meta_answer_type"].value_counts())

meta_intent
Informational    59
Data/List        20
Fact-Seeking     18
Name: count, dtype: int64 

meta_answer_type
Narrative    47
Atomic       34
Complex      16
Name: count, dtype: int64


## Classical QE techniques

In [3]:
import re

def clean_query(q):
    q = re.sub(r"[^\w\s]", " ", str(q))
    q = re.sub(r"\s+", " ", q).strip()
    return q

# Clean queries (else it gives errors in PyTerrier)
topics["query"] = topics["query"].apply(clean_query)

# Initialize BM25 retriever
bm25 = pt.terrier.Retriever(index, wmodel="BM25")

# Initialize classic query expansion models
rm3 = pt.rewrite.RM3(index, fb_terms=10, fb_docs=3)
bo1 = pt.rewrite.Bo1QueryExpansion(index, fb_terms=10, fb_docs=3)

bm25_rm3 = bm25 >> rm3 >> bm25
bm25_bo1 = bm25 >> bo1 >> bm25

Java started (triggered by Retriever.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


## LLM-based QE techniques 

In [4]:
import sys
!{sys.executable} -m pip install ollama

'd:\MSc' is not recognized as an internal or external command,
operable program or batch file.


## Query2doc

In [5]:
import ollama
import pandas as pd
from tqdm import tqdm
import os


file_path = "query2doc/topics_with_q2d.csv"

if os.path.exists(file_path):
    q2d_topics = pd.read_csv(file_path)
    q2d_topics["qid"] = q2d_topics["qid"].astype(str)
    q2d_topics["q2d_query"] = q2d_topics["q2d_query"].apply(clean_query)

else:
    print("Generating query2doc documents...")

    def expand_query_sparse(original_query, llm_generated_doc):
        # Paper recommends boosting the original query by appending it 5 times, followed by the generated pseudo-document.
        boosted_query = (original_query + " ") * 5
        return boosted_query + llm_generated_doc

    def generate_q2d_query(query):

        system_msg = "You are a specialized query expansion assistant. Your ONLY task is to write a single informative passage that answers the user's target query. Do not provide multiple answers or refer to previous examples."
        # Generate a pseudo-document using the original query as input to the LLM.
        # We use a few-shot prompt to guide the LLM in generating a passage that is relevant to the query.
        # The examples are directly taken from the paper.
        prompt = f"""### Instruction ### Write a passage that answers the given query:

        ### Examples ###
        Query: what state is this zip code 85282
        Passage: Welcome to TEMPE, AZ 85282. 85282 is a rural zip code in Tempe, Arizona. The population
        is primarily white, and mostly single. At $200,200 the average home value here is a bit higher than
        average for the Phoenix-Mesa-Scottsdale metro area, so this probably isn’t the place to look for housing
        bargains.5282 Zip code is located in the Mountain time zone at 33 degrees latitude (Fun Fact: this is the
        same latitude as Damascus, Syria!) and -112 degrees longitude.

        Query: why is gibbs model of reflection good
        Passage: In this reflection, I am going to use Gibbs (1988) Reflective Cycle. This model is a recognised
        framework for my reflection. Gibbs (1988) consists of six stages to complete one cycle which is able
        to improve my nursing practice continuously and learning from the experience for better practice in the
        future.n conclusion of my reflective assignment, I mention the model that I chose, Gibbs (1988) Reflective
        Cycle as my framework of my reflective. I state the reasons why I am choosing the model as well as some
        discussion on the important of doing reflection in nursing practice.

        Query: what does a thousand pardons means
        Passage: Oh, that’s all right, that’s all right, give us a rest; never mind about the direction, hang the
        direction - I beg pardon, I beg a thousand pardons, I am not well to-day; pay no attention when I soliloquize,
        it is an old habit, an old, bad habit, and hard to get rid of when one’s digestion is all disordered with eating
        food that was raised forever and ever before he was born; good land! a man can’t keep his functions
        regular on spring chickens thirteen hundred years old.

        Query: what is a macro warning
        Passage: Macro virus warning appears when no macros exist in the file in Word. When you open
        a Microsoft Word 2002 document or template, you may receive the following macro virus warning,
        even though the document or template does not contain macros: C:\<path>\<file name>contains macros.
        Macros may contain viruses.
        
        ### Task ###
        Query: {query}
        Passage:"""

        response = ollama.generate(
            model="llama3.1",
            prompt=prompt,
            system=system_msg,
            options={
                "stop": [
                    "Query:",
                    "###",
                ],  
                "num_predict": 150,  
            },
        )

        return expand_query_sparse(query, response["response"])

    # Since local generation takes time, use a progress bar
    tqdm.pandas()
    topics["q2d_query"] = topics["query"].progress_apply(generate_q2d_query)
    topics["q2d_query"] = (
        topics["q2d_query"].str.replace(r"\s+", " ", regex=True).str.strip()
    )

    q2d_topics = topics[["qid", "q2d_query"]]

    q2d_topics.to_csv("query2doc/topics_with_q2d.csv", index=False)
    print(f"Q2D queries generated and saved to {file_path}")

<>:89: SyntaxWarning: invalid escape sequence '\<'
<>:89: SyntaxWarning: invalid escape sequence '\<'
C:\Users\radha\AppData\Local\Temp\ipykernel_36152\1222949672.py:89: SyntaxWarning: invalid escape sequence '\<'


## Chain-of-thoughts

In [6]:
file_path = "cot/topics_with_cot.csv"

if os.path.exists(file_path):
    cot_topics = pd.read_csv(file_path)
    cot_topics["qid"] = cot_topics["qid"].astype(str)
    cot_topics["cot_query"] = cot_topics["cot_query"].apply(clean_query)
    print(f"Loading existing CoT expansions from {file_path}...")
else:
    print("Generating Chain-of-Thought expansions...")

    def generate_cot_query(query):

        system_msg = (
            "You are a retrieval assistant. Given a query, write a short factual passage "
            "with supporting context, then end with 'So the final answer is [answer].' "
            "Be concise. Do not use bullet points or step-by-step reasoning."
        )

        prompt = f"""Answer the following queries with a short factual passage followed by a final answer sentence.

        Query: who owns jaguar land rover?
        Passage: Jaguar Land Rover is a British multinational car manufacturer founded by William Lyons in 1931. Its headquarters are in Whitley, Coventry, United Kingdom. The company is a wholly owned subsidiary of Tata Motors of India. So the final answer is Tata Motors.

        Query: what year was the eiffel tower built?
        Passage: The Eiffel Tower is a wrought-iron lattice tower located on the Champ de Mars in Paris, France. It was designed by Gustave Eiffel and constructed between 1887 and 1889 as the entrance arch for the 1889 World's Fair. So the final answer is 1889.

        Query: what is the capital of australia?
        Passage: Australia is a federal parliamentary constitutional monarchy. Its capital city is Canberra, which was purpose-built as a compromise between rivals Sydney and Melbourne. Canberra became the capital in 1913. So the final answer is Canberra.

        Query: {query}
        Passage:"""

        response = ollama.generate(
            model="llama3.1",
            prompt=prompt,
            system=system_msg,
            options={
                "num_predict": 150,
            },
        )

        return query + response["response"]

    topics["cot_query"] = topics["query"].progress_apply(generate_cot_query)
    topics["cot_query"] = topics["cot_query"].apply(clean_query)

    cot_topics = topics[["qid", "cot_query"]]
    cot_topics.to_csv(file_path, index=False)
    print(f"CoT queries generated and saved to {file_path}")

Loading existing CoT expansions from cot/topics_with_cot.csv...


# Running the experiments

In [7]:
from pyterrier.measures import RR, nDCG, AP, R

if "q2d_query" in q2d_topics.columns:
    q2d_topics = q2d_topics.rename(columns={"q2d_query": "query"})

results_classical = pt.Experiment(
    [bm25, bm25_rm3, bm25_bo1],
    topics[["qid", "query"]],  
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=["BM25", "RM3", "Bo1"],
    perquery=True,
)

results_q2d = pt.Experiment(
    [bm25],
    q2d_topics[["qid", "query"]],
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=["Q2D_Llama3"],
    perquery=True,
)

results_cot = pt.Experiment(
    [bm25],
    cot_topics[["qid", "cot_query"]].rename(columns={"cot_query": "query"}),
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=["CoT_Llama3"],
    perquery=True,
)

results = pd.concat([results_classical, results_q2d, results_cot], ignore_index=True)


label_map = topics[["qid", "meta_intent", "meta_answer_type"]]
results = results.merge(label_map, on="qid", how="left")

# Overall results
print("\n=== Overall Results ===")
overall = results.groupby(["name", "measure"])["value"].mean().unstack()
print(overall)

# Results by query intent
print("\n=== nDCG@10 by Query Intent ===")
ndcg_by_intent = (
    results[results["measure"] == "nDCG@10"]
    .groupby(["name", "meta_intent"])["value"]
    .mean()
    .unstack()
)
print(ndcg_by_intent)

# Results by answer type
print("\n=== nDCG@10 by Answer Type ===")
ndcg_by_answer_type = (
    results[results["measure"] == "nDCG@10"]
    .groupby(["name", "meta_answer_type"])["value"]
    .mean()
    .unstack()
)
print(ndcg_by_answer_type)


=== Overall Results ===
measure     AP(rel=2)  R(rel=2)@100  RR(rel=2)@10   nDCG@10
name                                                       
BM25         0.290089      0.541421      0.625749  0.487382
Bo1          0.311664      0.568762      0.618295  0.500851
CoT_Llama3   0.338962      0.559121      0.643941  0.514828
Q2D_Llama3   0.389330      0.628159      0.756554  0.578139
RM3          0.317579      0.575153      0.628371  0.516954

=== nDCG@10 by Query Intent ===
meta_intent  Data/List  Fact-Seeking  Informational
name                                               
BM25          0.462645      0.496273       0.493056
Bo1           0.516190      0.494730       0.497519
CoT_Llama3    0.505907      0.481448       0.528036
Q2D_Llama3    0.560193      0.573080       0.585766
RM3           0.505432      0.514619       0.521572

=== nDCG@10 by Answer Type ===
meta_answer_type    Atomic   Complex  Narrative
name                                           
BM25              0.510066  0.

In [8]:
!pip install matplotlib 


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
metric = "nDCG@10"

# Pivot to get one column per system per query
per_query = (
    results[results["measure"] == metric]
    .pivot_table(index="qid", columns="name", values="value")
    .reset_index()
)

# Best classical vs best LLM
per_query["best_classical"] = per_query[["RM3", "Bo1"]].max(axis=1)
per_query["best_classical_name"] = per_query[["RM3", "Bo1"]].idxmax(axis=1)
per_query["best_llm"] = per_query[["Q2D_Llama3", "CoT_Llama3"]].max(axis=1)
per_query["best_llm_name"] = per_query[["Q2D_Llama3", "CoT_Llama3"]].idxmax(axis=1)
per_query["classical_advantage"] = per_query["best_classical"] - per_query["best_llm"]

# Merge in query text and annotations
query_meta = topics[["qid", "query", "meta_intent", "meta_answer_type"]]
per_query = per_query.merge(query_meta, on="qid", how="left")

# Queries where classical beat BOTH LLM methods
classical_wins = (
    per_query[
        (per_query["RM3"] > per_query["Q2D_Llama3"])
        & (per_query["RM3"] > per_query["CoT_Llama3"])
        | (per_query["Bo1"] > per_query["Q2D_Llama3"])
        & (per_query["Bo1"] > per_query["CoT_Llama3"])
    ]
    .sort_values("classical_advantage", ascending=False)
    .head(10)[
        [
            "query",
            "best_classical_name",
            "best_classical",
            "best_llm_name",
            "best_llm",
            "classical_advantage",
            "meta_intent",
            "meta_answer_type",
        ]
    ]
    .rename(
        columns={
            "best_classical_name": "classical_method",
            "best_classical": f"classical_{metric}",
            "best_llm_name": "llm_method",
            "best_llm": f"llm_{metric}",
            "classical_advantage": "advantage",
        }
    )
    .reset_index(drop=True)
)

print(f"Top queries where Classical QE outperformed BOTH LLM methods ({metric}):\n")
print(
    f"Total: {len(per_query[(per_query['RM3'] > per_query['Q2D_Llama3']) & (per_query['RM3'] > per_query['CoT_Llama3']) | (per_query['Bo1'] > per_query['Q2D_Llama3']) & (per_query['Bo1'] > per_query['CoT_Llama3'])])} out of {len(per_query)} queries\n"
)
display(
    classical_wins.style.format(
        {
            f"classical_{metric}": "{:.3f}",
            f"llm_{metric}": "{:.3f}",
            "advantage": "{:.3f}",
        }
    )
)

Top queries where Classical QE outperformed BOTH LLM methods (nDCG@10):

Total: 30 out of 97 queries



,query,classical_method,classical_nDCG@10,llm_method,llm_nDCG@10,advantage,meta_intent,meta_answer_type
0,is caffeine an narcotic,RM3,0.842,Q2D_Llama3,0.000,0.842,Fact-Seeking,Atomic
1,what can you do about discrimination in the workplace in oklahoma city,RM3,0.756,Q2D_Llama3,0.255,0.501,Informational,Narrative
2,dog day afternoon meaning,RM3,0.496,Q2D_Llama3,0.156,0.340,Informational,Narrative
3,average wedding dress alteration cost,RM3,0.856,Q2D_Llama3,0.558,0.299,Data/List,Atomic
4,what is wifi vs bluetooth,Bo1,0.371,Q2D_Llama3,0.081,0.290,Informational,Complex
5,how many sons robert kraft has,RM3,0.759,Q2D_Llama3,0.469,0.289,Fact-Seeking,Atomic
6,meaning of shebang,RM3,0.952,Q2D_Llama3,0.698,0.254,Fact-Seeking,Narrative
7,what is the most popular food in switzerland,RM3,0.870,Q2D_Llama3,0.630,0.240,Informational,Atomic
8,what is a statutory deed,RM3,0.944,CoT_Llama3,0.746,0.198,Informational,Narrative
9,what does it mean if your tsh is low,RM3,0.552,Q2D_Llama3,0.367,0.185,Informational,Atomic


### Inspecting the queries 

In [10]:
for answer_type in annotations["answer_type"].unique():
    examples = (
        annotations[annotations["answer_type"] == answer_type]["query"].head(3).tolist()
    )
    count = (annotations["answer_type"] == answer_type).sum()
    print(f"\n{str(answer_type).upper()} ({count} queries):")
    for ex in examples:
        print(f"  - {ex}")


LONG ANSWER (37 queries):
  - benefit policy in layoff
  - what is onboarding for credit unions
  - how much money do motivational speakers make

SHORT ANSWER (30 queries):
  - what is an aml surveillance analyst
  - who owns barnhart crane
  - how much does talent directors get paid a year

COMPARISON (5 queries):
  - difference between a company's strategy and business model is
  - difference between a mcdouble and a double cheeseburger
  - difference between rn and bsn

LIST (30 queries):
  - what types of food can you cook sous vide
  - what is supplemental security income used for
  - foods to detox liver naturally

FACTOID (81 queries):
  - average salary for dental hygienist in nebraska
  - how long does it take to get refund from petsmart
  - how much weight on usps letter

SHORT DESCRIPTION (8 queries):
  - who is aziz hashim
  - why did the ancient egyptians call their land kemet, or black land?
  - what type of conflict does della face in o, henry the gift of the magi

DEFI

In [11]:
for intent_type in annotations["intent"].unique():
    examples = (
        annotations[annotations["intent"] == intent_type]["query"].head(3).tolist()
    )
    count = (annotations["intent"] == intent_type).sum()
    print(f"\n{intent_type.upper()} ({count} queries):")
    for ex in examples:
        print(f"  - {ex}")


DESCRIPTION (157 queries):
  - benefit policy in layoff
  - what is onboarding for credit unions
  - what is an aml surveillance analyst

ENTITY (30 queries):
  - who owns barnhart crane
  - who sings monk theme song
  - what language is craith filmed in?

LIST (48 queries):
  - what types of food can you cook sous vide
  - how to get a free xbox one card
  - where is the show shameless filmed

QUANTITY (52 queries):
  - average salary for dental hygienist in nebraska
  - how long does it take to get refund from petsmart
  - how much does talent directors get paid a year

ATTRIBUTE (25 queries):
  - what is reba mcentire's net worth
  - barclays fca number
  - routing number for savings bank of maine

TEMPORAL (6 queries):
  - what year did knee deep come out funkadelic
  - landfill hours
  - when is the evening

PROCESS (9 queries):
  - what temperature and humidity to dry sausage
  - how do they do open heart surgery
  - what the best way to get clothes white

LOCATION (12 queries):

## Analyzing queries based on query difficulty

In [12]:
hard_topics = pd.read_csv(
    "https://raw.githubusercontent.com/grill-lab/DL-Hard/main/dataset/topics.tsv",
    sep="\t",
    header=None,
    names=["qid", "query"],
)
hard_qids = set(hard_topics["qid"].astype(str))
dl_hard = pt.get_dataset("irds:msmarco-passage/trec-dl-hard")
hard_qrels = dl_hard.get_qrels()
hard_qrels["qid"] = hard_qrels["qid"].astype(str)

print(hard_qrels.shape)
print(hard_qrels["qid"].nunique())

all_qrels = pd.concat([dl19.get_qrels(), dl20.get_qrels(), hard_qrels])
all_qrels["qid"] = all_qrels["qid"].astype(str)

qrels_qids = set(all_qrels["qid"].unique())
topics = annotations[annotations["qid"].isin(qrels_qids)].reset_index(drop=True)

hard_qids = set(hard_topics["qid"].astype(str))
topics["is_hard"] = topics["qid"].isin(hard_qids)

print(f"Hard queries: {topics['is_hard'].sum()}")

(4256, 4)
50
Hard queries: 50


In [13]:
from pyterrier.measures import RR, nDCG, AP, R

if "q2d_query" in q2d_topics.columns:
    q2d_topics = q2d_topics.rename(columns={"q2d_query": "query"})

# Split topics into hard and non-hard
hard_topics_subset    = topics[topics["is_hard"]][["qid", "query"]].copy()
nonhard_topics_subset = topics[~topics["is_hard"]][["qid", "query"]].copy()

hard_topics_subset["query"]    = hard_topics_subset["query"].apply(clean_query)
nonhard_topics_subset["query"] = nonhard_topics_subset["query"].apply(clean_query)

dl_hard_qrels = hard_qrels.copy()
dl_hard_qrels["qid"] = dl_hard_qrels["qid"].astype(str)

nonhard_qrels = pd.concat([dl19.get_qrels(), dl20.get_qrels()])
nonhard_qrels["qid"] = nonhard_qrels["qid"].astype(str)
nonhard_qrels = nonhard_qrels[nonhard_qrels["qid"].isin(set(nonhard_topics_subset["qid"]))]

# Run experiments separately for hard and non-hard groups.
# Note: q2d_topics already has column "query" (renamed in Cell 13).
# cot_topics uses column "cot_query" so we rename inline.
def run_experiment(topics_df, qrels_df, q2d_df, cot_df, label):
    classical = pt.Experiment(
        [bm25, bm25_rm3, bm25_bo1],
        topics_df,
        qrels_df,
        eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
        names=["BM25", "RM3", "Bo1"],
        perquery=True,
    )

    q2d_subset = q2d_df[q2d_df["qid"].isin(set(topics_df["qid"]))].copy()
    q2d_res = pt.Experiment(
        [bm25],
        q2d_subset[["qid", "query"]],
        qrels_df,
        eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
        names=["Q2D_Llama3"],
        perquery=True,
    )

    cot_subset = cot_df[cot_df["qid"].isin(set(topics_df["qid"]))].copy()
    cot_res = pt.Experiment(
        [bm25],
        cot_subset[["qid", "cot_query"]].rename(columns={"cot_query": "query"}),
        qrels_df,
        eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
        names=["CoT_Llama3"],
        perquery=True,
    )

    combined = pd.concat([classical, q2d_res, cot_res], ignore_index=True)
    combined["group"] = label
    return combined

hard_results    = run_experiment(hard_topics_subset,    dl_hard_qrels, q2d_topics, cot_topics, "Hard")
nonhard_results = run_experiment(nonhard_topics_subset, nonhard_qrels, q2d_topics, cot_topics, "Non-Hard")

all_results = pd.concat([hard_results, nonhard_results], ignore_index=True)

print("=== Results: Hard vs. Non-Hard ===")
pivot = all_results.pivot_table(
    index="name", columns=["measure", "group"], values="value"
)
print(pivot.to_string())


=== Results: Hard vs. Non-Hard ===
measure    AP(rel=2)           R(rel=2)@100           RR(rel=2)@10             nDCG@10          
group           Hard  Non-Hard         Hard  Non-Hard         Hard  Non-Hard      Hard  Non-Hard
name                                                                                            
BM25        0.147106  0.313310     0.454211  0.572496     0.415056  0.598543  0.274333  0.498014
Bo1         0.158566  0.336298     0.450608  0.609229     0.398722  0.592502  0.266630  0.518096
CoT_Llama3  0.225974  0.370182     0.405575  0.601549     0.611111  0.653013  0.376173  0.553141
Q2D_Llama3  0.267312  0.423046     0.483685  0.668080     0.760771  0.755388  0.467670  0.608664
RM3         0.164574  0.341393     0.478356  0.608999     0.409579  0.605582  0.284258  0.529642


In [14]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# NOTE: variable named "difficulty_rows" (not "results") to avoid
# overwriting the results DataFrame from Cell 13.
difficulty_rows = []

methods_diff  = all_results["name"].unique()
measures_diff = all_results["measure"].unique()

for method in methods_diff:
    for measure in measures_diff:
        subset = all_results[
            (all_results["name"] == method) & (all_results["measure"] == measure)
        ]
        hard_scores    = subset[subset["group"] == "Hard"]["value"]
        nonhard_scores = subset[subset["group"] == "Non-Hard"]["value"]

        stat, p = mannwhitneyu(hard_scores, nonhard_scores, alternative="two-sided")

        difficulty_rows.append({
            "method":       method,
            "measure":      measure,
            "p_value":      p,
            "hard_mean":    hard_scores.mean(),
            "nonhard_mean": nonhard_scores.mean(),
        })

difficulty_df = pd.DataFrame(difficulty_rows)
print("=== Mann-Whitney U: Hard vs Non-Hard (raw p-values) ===")
print(difficulty_df.sort_values("p_value"))


=== Mann-Whitney U: Hard vs Non-Hard (raw p-values) ===
        method       measure   p_value  hard_mean  nonhard_mean
4          Bo1       nDCG@10  0.000007   0.266630      0.518096
8          RM3       nDCG@10  0.000013   0.284258      0.529642
0         BM25       nDCG@10  0.000020   0.274333      0.498014
5          Bo1     AP(rel=2)  0.000022   0.158566      0.336298
9          RM3     AP(rel=2)  0.000055   0.164574      0.341393
1         BM25     AP(rel=2)  0.000111   0.147106      0.313310
11         RM3  RR(rel=2)@10  0.006316   0.409579      0.605582
7          Bo1  RR(rel=2)@10  0.008142   0.398722      0.592502
3         BM25  RR(rel=2)@10  0.008423   0.415056      0.598543
6          Bo1  R(rel=2)@100  0.015573   0.450608      0.609229
16  CoT_Llama3       nDCG@10  0.018536   0.376173      0.553141
18  CoT_Llama3  R(rel=2)@100  0.022794   0.405575      0.601549
13  Q2D_Llama3     AP(rel=2)  0.023560   0.267312      0.423046
17  CoT_Llama3     AP(rel=2)  0.024099   0.22597

In [15]:
# Multiple testing correction using Holm-Bonferroni method

reject, pvals_corrected, _, _ = multipletests(difficulty_df["p_value"], method="holm")

difficulty_df["p_corrected"] = pvals_corrected
difficulty_df["significant"]  = reject

print("=== Mann-Whitney U: Hard vs Non-Hard (Holm-corrected) ===")
print(difficulty_df.sort_values("p_corrected"))


=== Mann-Whitney U: Hard vs Non-Hard (Holm-corrected) ===
        method       measure   p_value  hard_mean  nonhard_mean  p_corrected  \
4          Bo1       nDCG@10  0.000007   0.266630      0.518096     0.000140   
8          RM3       nDCG@10  0.000013   0.284258      0.529642     0.000255   
0         BM25       nDCG@10  0.000020   0.274333      0.498014     0.000355   
5          Bo1     AP(rel=2)  0.000022   0.158566      0.336298     0.000370   
9          RM3     AP(rel=2)  0.000055   0.164574      0.341393     0.000886   
1         BM25     AP(rel=2)  0.000111   0.147106      0.313310     0.001669   
11         RM3  RR(rel=2)@10  0.006316   0.409579      0.605582     0.088429   
7          Bo1  RR(rel=2)@10  0.008142   0.398722      0.592502     0.105848   
3         BM25  RR(rel=2)@10  0.008423   0.415056      0.598543     0.105848   
6          Bo1  R(rel=2)@100  0.015573   0.450608      0.609229     0.171304   
16  CoT_Llama3       nDCG@10  0.018536   0.376173      0.55314

In [16]:
from itertools import combinations

# =============================================================
# STATISTICAL TESTS — QUERY INTENT
# =============================================================
from scipy.stats import kruskal

def run_kruskal_pairwise(results_df, groupby_col, methods, measures):
    """
    For each (method, measure) pair:
      1. Kruskal-Wallis across all groups (omnibus test — is there ANY difference?)
      2. Pairwise Mann-Whitney U between every pair of groups (which pairs differ?)
    Holm-Bonferroni correction applied across all pairwise p-values.
    """
    kruskal_rows = []
    pairwise_rows = []
    groups = results_df[groupby_col].dropna().unique()

    for method in methods:
        for measure in measures:
            subset = results_df[
                (results_df["name"] == method) & (results_df["measure"] == measure)
            ]
            group_scores = {
                g: subset[subset[groupby_col] == g]["value"].values
                for g in groups
            }
            valid_groups = [g for g, s in group_scores.items() if len(s) >= 2]
            if len(valid_groups) < 2:
                continue

            kw_stat, kw_p = kruskal(*[group_scores[g] for g in valid_groups])
            kruskal_rows.append({
                "method": method,
                "measure": measure,
                "kw_statistic": round(kw_stat, 4),
                "kw_p_value": round(kw_p, 4),
                "group_means": {g: round(group_scores[g].mean(), 4) for g in valid_groups},
            })

            for g1, g2 in combinations(valid_groups, 2):
                s1, s2 = group_scores[g1], group_scores[g2]
                stat, p = mannwhitneyu(s1, s2, alternative="two-sided")
                pairwise_rows.append({
                    "method": method,
                    "measure": measure,
                    "group_1": g1,
                    "group_2": g2,
                    "mw_statistic": round(stat, 4),
                    "p_value": p,
                    "mean_group_1": round(s1.mean(), 4),
                    "mean_group_2": round(s2.mean(), 4),
                })

    kruskal_df = pd.DataFrame(kruskal_rows)
    pairwise_df = pd.DataFrame(pairwise_rows)

    if len(pairwise_df) > 0:
        reject, p_corr, _, _ = multipletests(pairwise_df["p_value"], method="holm")
        pairwise_df["p_corrected"] = p_corr.round(4)
        pairwise_df["significant"] = reject

    return kruskal_df, pairwise_df


# Re-derive methods/measures from the results dataframe in Cell 13
methods_qt   = results["name"].unique()
measures_qt  = results["measure"].unique()

# --- Query Intent ---
kw_intent, pw_intent = run_kruskal_pairwise(results, "meta_intent", methods_qt, measures_qt)
print("=== Kruskal-Wallis: Query Intent ===")
print(kw_intent.to_string(index=False))
print("=== Pairwise Mann-Whitney (Holm-corrected): Query Intent ===")
print(pw_intent.sort_values("p_corrected").to_string(index=False))

# --- Answer Type ---
kw_ans, pw_ans = run_kruskal_pairwise(results, "meta_answer_type", methods_qt, measures_qt)
print("=== Kruskal-Wallis: Answer Type ===")
print(kw_ans.to_string(index=False))
print("=== Pairwise Mann-Whitney (Holm-corrected): Answer Type ===")
print(pw_ans.sort_values("p_corrected").to_string(index=False))

# =============================================================
# STATISTICAL TESTS — DIFFICULTY: Hard vs. Non-Hard per method
# (already in Cells 23-24, reproduced here cleanly with stat column)
# =============================================================
print("" + "="*60)
print("DIFFICULTY: per-method Hard vs Non-Hard")
print("="*60)
diff_rows = []
methods_diff   = all_results["name"].unique()
measures_diff  = all_results["measure"].unique()

for method in methods_diff:
    for measure in measures_diff:
        subset = all_results[
            (all_results["name"] == method) & (all_results["measure"] == measure)
        ]
        hard_scores    = subset[subset["group"] == "Hard"]["value"].values
        nonhard_scores = subset[subset["group"] == "Non-Hard"]["value"].values
        if len(hard_scores) < 2 or len(nonhard_scores) < 2:
            continue
        stat, p = mannwhitneyu(hard_scores, nonhard_scores, alternative="two-sided")
        diff_rows.append({
            "method":       method,
            "measure":      measure,
            "hard_mean":    round(hard_scores.mean(), 4),
            "nonhard_mean": round(nonhard_scores.mean(), 4),
            "delta":        round(nonhard_scores.mean() - hard_scores.mean(), 4),
            "mw_stat":      round(stat, 4),
            "p_value":      p,
        })

diff_df = pd.DataFrame(diff_rows)
reject, p_corr, _, _ = multipletests(diff_df["p_value"], method="holm")
diff_df["p_corrected"] = p_corr.round(4)
diff_df["significant"]  = reject
print(diff_df.sort_values(["measure", "p_corrected"]).to_string(index=False))

# =============================================================
# STATISTICAL TESTS — METHOD vs. METHOD within Hard / Non-Hard
# =============================================================
print("" + "="*60)
print("METHOD vs. METHOD within Hard and Non-Hard groups")
print("="*60)
# This answers the core RQ: among hard queries, is Q2D significantly
# better or worse than RM3/Bo1? Same question for non-hard queries.

method_comp_rows = []

for group_label in ["Hard", "Non-Hard"]:
    group_df = all_results[all_results["group"] == group_label]

    for measure in measures_diff:
        measure_df = group_df[group_df["measure"] == measure]

        for m1, m2 in combinations(methods_diff, 2):
            s1 = measure_df[measure_df["name"] == m1]["value"].values
            s2 = measure_df[measure_df["name"] == m2]["value"].values
            if len(s1) < 2 or len(s2) < 2:
                continue
            stat, p = mannwhitneyu(s1, s2, alternative="two-sided")
            method_comp_rows.append({
                "group":    group_label,
                "measure":  measure,
                "method_1": m1,
                "method_2": m2,
                "mean_m1":  round(s1.mean(), 4),
                "mean_m2":  round(s2.mean(), 4),
                "delta":    round(s1.mean() - s2.mean(), 4),  # positive = m1 better
                "mw_stat":  round(stat, 4),
                "p_value":  p,
            })

method_comp_df = pd.DataFrame(method_comp_rows)
reject, p_corr, _, _ = multipletests(method_comp_df["p_value"], method="holm")
method_comp_df["p_corrected"] = p_corr.round(4)
method_comp_df["significant"]  = reject

print("--- HARD queries ---")
print(
    method_comp_df[method_comp_df["group"] == "Hard"]
    .sort_values(["measure", "p_corrected"])
    [["measure","method_1","method_2","mean_m1","mean_m2","delta","p_value","p_corrected","significant"]]
    .to_string(index=False)
)

print("--- NON-HARD queries ---")
print(
    method_comp_df[method_comp_df["group"] == "Non-Hard"]
    .sort_values(["measure", "p_corrected"])
    [["measure","method_1","method_2","mean_m1","mean_m2","delta","p_value","p_corrected","significant"]]
    .to_string(index=False)
)

print("--- Significant differences only (p_corrected < 0.05) ---")
sig = method_comp_df[method_comp_df["significant"]].sort_values(["group","measure","p_corrected"])
if len(sig) == 0:
    print("No significant pairwise method differences found after Holm-Bonferroni correction.")
else:
    print(sig[["group","measure","method_1","method_2","mean_m1","mean_m2","delta","p_corrected"]].to_string(index=False))


=== Kruskal-Wallis: Query Intent ===
    method      measure  kw_statistic  kw_p_value                                                            group_means
      BM25      nDCG@10        0.1303      0.9369 {'Informational': 0.4931, 'Fact-Seeking': 0.4963, 'Data/List': 0.4626}
      BM25    AP(rel=2)        1.3474      0.5098  {'Informational': 0.2859, 'Fact-Seeking': 0.362, 'Data/List': 0.2376}
      BM25 R(rel=2)@100        6.3547      0.0417 {'Informational': 0.5318, 'Fact-Seeking': 0.6982, 'Data/List': 0.4287}
      BM25 RR(rel=2)@10        0.3541      0.8377 {'Informational': 0.6331, 'Fact-Seeking': 0.5761, 'Data/List': 0.6489}
       Bo1      nDCG@10        0.0825      0.9596 {'Informational': 0.4975, 'Fact-Seeking': 0.4947, 'Data/List': 0.5162}
       Bo1    AP(rel=2)        1.6806      0.4316 {'Informational': 0.3037, 'Fact-Seeking': 0.3883, 'Data/List': 0.2661}
       Bo1 R(rel=2)@100        8.2143      0.0165 {'Informational': 0.5624, 'Fact-Seeking': 0.7335, 'Data/List': 0.4